# Notebook 1 — Setup, Prompt Registry, and Golden Dataset

 Purpose:
 - validate environment and provider configuration
 - initialize MLflow tracking
 - register the baseline prompt in MLflow Prompt Registry
 - create the reference evaluation dataset in MLflow

 This notebook does NOT:
 - run model inference
 - log generation traces
 - evaluate outputs

In [ ]:
# %pip install -r ../requirements.txt


In [1]:
from __future__ import annotations

import os
import sys
import json
from pathlib import Path
from typing import Any

import mlflow
import openai
import pandas as pd
from dotenv import load_dotenv

In [2]:
# Robust project root detection
PROJECT_ROOT = Path.cwd()

# Adjust if notebook is inside /notebooks
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

ENV_PATH = PROJECT_ROOT / ".env"
REQUIREMENTS_PATH = PROJECT_ROOT / "requirements.txt"

print("PROJECT_ROOT:", PROJECT_ROOT)
print(".env exists:", ENV_PATH.exists())
print("requirements.txt exists:", REQUIREMENTS_PATH.exists())

PROJECT_ROOT: c:\Users\BRHN\Desktop\SLM-evals
.env exists: True
requirements.txt exists: True


# Load environment variables

In [3]:
load_dotenv(ENV_PATH)

# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

PHI4_API_KEY = os.getenv("PHI4_API_KEY")
PHI4_ENDPOINT = os.getenv("PHI4_ENDPOINT")
PHI4_DEPLOYMENT = os.getenv("PHI4_DEPLOYMENT")

# Llama 3.1 instead 
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
BASE_URL = os.getenv("BASE_URL")
GROQ_LLAMA_MODEL = os.getenv("GROQ_LLAMA_MODEL")



MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_EXPERIMENT_NAME = os.getenv("MLFLOW_EXPERIMENT_NAME")

In [ ]:
required_env = {
    #"OPENAI_API_KEY": OPENAI_API_KEY,
    "PHI4_API_KEY": PHI4_API_KEY,
    "PHI4_ENDPOINT": PHI4_ENDPOINT,
    "PHI4_DEPLOYMENT": PHI4_DEPLOYMENT,
    "GROQ_API_KEY": GROQ_API_KEY,
    "BASE_URL": BASE_URL,
    "GROQ_LLAMA_MODEL": GROQ_LLAMA_MODEL,
    "MLFLOW_TRACKING_URI": MLFLOW_TRACKING_URI,
    "MLFLOW_EXPERIMENT_NAME": MLFLOW_EXPERIMENT_NAME,
}

print("Environment validation:")
for key, value in required_env.items():
    print(f" - {key}: {'OK' if bool(value) else 'MISSING'}")

missing = [key for key, value in required_env.items() if not value]
if missing:
    raise EnvironmentError(
        "Missing required environment variables: " + ", ".join(missing)
    )

In [4]:
print("Python version:", sys.version)
print("MLflow version:", mlflow.__version__)
print("OpenAI version:", openai.__version__)
print("Pandas version:", pd.__version__)

Python version: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)]
MLflow version: 3.10.1
OpenAI version: 2.29.0
Pandas version: 2.3.3


# Configure MLflow tracking and experiment

In [5]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

experiment = mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if experiment is None:
    raise RuntimeError("MLflow experiment could not be created or resolved.")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)

Tracking URI: http://127.0.0.1:5000
Experiment name: phi4_vs_Llama3-1_baseline
Experiment ID: 4


* Notebook 1 run for bootstrap lineage

In [6]:
with mlflow.start_run(run_name="nb2_bootstrap_setup") as run:
    mlflow.log_params({
        "notebook_role": "setup_prompt_dataset",
        "experiment_name": MLFLOW_EXPERIMENT_NAME,
        "project_root": str(PROJECT_ROOT),
    })
    
    mlflow.log_dict(
        {
            "python": sys.version,
            "mlflow": mlflow.__version__,
            "openai": openai.__version__,
            "pandas": pd.__version__,
        },
        "environment/package_versions.json",
    )

    BOOTSTRAP_RUN_ID = run.info.run_id

print("Bootstrap run ID:", BOOTSTRAP_RUN_ID)

🏃 View run nb2_bootstrap_setup at: http://127.0.0.1:5000/#/experiments/4/runs/e7eb046d032a409fa8f3404047bddea4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
Bootstrap run ID: e7eb046d032a409fa8f3404047bddea4


# Define the baseline prompt

In [7]:
PROMPT_NAME = "financial_analysis_baseline"

PROMPT_TEMPLATE = [
    {
        "role": "system",
        "content": (
            "You are a financial analyst specialized in bank performance analysis.\n\n"
            "Your task is to analyze only the financial data provided by the user and write a professional, grounded, and well-structured financial analysis report.\n\n"
            "Strict rules:\n"
            "1. Base your answer only on the provided data.\n"
            "2. Do not invent numbers, ratios, trends, explanations, or unsupported claims.\n"
            "3. Do not rename, merge, omit, or reorder section titles.\n"
            "4. Use exactly the following section titles, in exactly this order:\n"
                "   Executive Summary\n"
                "   Profitability and Operational Efficiency\n"
                "   Revenue Dynamics\n"
                "   Asset Quality and Risk Profile\n"
                "   Balance Sheet Structure and Liquidity\n"
                "   Capital Adequacy\n"
                "   Key Risks and Watch Points\n"
                "   Conclusion\n"
            "5. Every section must appear in the final answer, even if brief.\n"
            "6. If the provided data is insufficient for a section, write exactly: Not enough information in the provided data.\n"
            "7. Use concise analytical paragraphs under each section title.\n"
            "8. Focus on factual interpretation, financial trends, balance sheet quality, risk indicators, capital strength, and internal consistency.\n"
            "9. Output only the final report, with no preface, no notes, no bullet list of rules, and no extra commentary.\n\n"
            "Formatting requirements:\n"
            "- Write each section title exactly as provided above.\n"
            "- Put each section title on its own line.\n"
            "- Put the section content immediately below it.\n"
            "- Do not use markdown bullets unless the section naturally requires a short list."
        ),
    },
    {
        "role": "user",
        "content": (
            "Analyze the following bank financial data and write a structured financial analysis report.\n\n"
            "Financial data:\n\n"
            "{{financial_data}}"
        ),
    },
]

# Register the prompt in MLflow

In [8]:
prompt_version = mlflow.genai.register_prompt(
    name=PROMPT_NAME,
    template=PROMPT_TEMPLATE,
    commit_message="Initial baseline prompt for bank financial analysis benchmark",
    tags={
        "project": "slm-evals",
        "task": "bank_financial_analysis",
        "stage": "baseline",
    },
)

print("Prompt name:", prompt_version.name)
print("Prompt version:", prompt_version.version)

2026/04/25 22:45:35 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: financial_analysis_baseline, version 10


Prompt name: financial_analysis_baseline
Prompt version: 10


# Create stable aliases

In [9]:
mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="baseline",
    version=prompt_version.version,
)

mlflow.genai.set_prompt_alias(
    name=PROMPT_NAME,
    alias="nb1_current",
    version=prompt_version.version,
)

PROMPT_URI_BASELINE = f"prompts:/{PROMPT_NAME}@baseline"
PROMPT_URI_VERSIONED = f"prompts:/{PROMPT_NAME}/{prompt_version.version}"

print("Baseline prompt URI:", PROMPT_URI_BASELINE)
print("Versioned prompt URI:", PROMPT_URI_VERSIONED)

Baseline prompt URI: prompts:/financial_analysis_baseline@baseline
Versioned prompt URI: prompts:/financial_analysis_baseline/10


# Load prompt back and validate

In [10]:
loaded_prompt = mlflow.genai.load_prompt(PROMPT_URI_BASELINE)

print("Loaded prompt name:", loaded_prompt.name)
print("Loaded prompt version:", loaded_prompt.version)
print("Is text prompt:", loaded_prompt.is_text_prompt)

Loaded prompt name: financial_analysis_baseline
Loaded prompt version: 10
Is text prompt: False


# Define the golden BIAT input

In [11]:
financial_input_text = """
# Financial Analysis – Banque Internationale Arabe de Tunisie (BIAT)

## Income Statement (TND millions)

Metric | 2021 | 2022 | 2023 | 2024
---|---|---|---|---
Net Banking Income (NBI) | 1050 | 1180 | 1320 | 1450
Operating Expenses | 520 | 560 | 610 | 660
Cost of Risk | 150 | 140 | 135 | 140
Net Income | 280 | 330 | 390 | 420

## Balance Sheet (TND millions)

Metric | 2021 | 2022 | 2023 | 2024
---|---|---|---|---
Total Assets | 18000 | 19500 | 21000 | 22500
Loans | 12000 | 13200 | 14500 | 15800
Deposits | 13500 | 14800 | 16200 | 17500
Equity | 1800 | 2000 | 2200 | 2400

## Risk & Capital Indicators

Metric | 2021 | 2022 | 2023 | 2024
---|---|---|---|---
NPL Ratio (%) | 8.5 | 8.0 | 7.6 | 7.8
Coverage Ratio (%) | 65 | 68 | 70 | 69
CET1 Ratio (%) | 11.5 | 12.0 | 12.3 | 12.5
LCR (%) | 120 | 125 | 130 | 128

## Key Financial Ratios (Derived)

### Profitability

Metric | 2021 | 2022 | 2023 | 2024
---|---|---|---|---
ROE (%) | 15.6 | 16.5 | 17.7 | 17.5
ROA (%) | 1.56 | 1.69 | 1.86 | 1.87
Cost-to-Income (%) | 49.5 | 47.5 | 46.2 | 45.5

### Growth

Metric | 2021 | 2022 | 2023 | 2024
---|---|---|---|---
NBI Growth (%) | - | 12.4 | 11.9 | 9.8
Loan Growth (%) | - | 10.0 | 9.8 | 9.0
Deposit Growth (%) | - | 9.6 | 9.5 | 8.0

### Risk Metrics

Metric | 2021 | 2022 | 2023 | 2024
---|---|---|---|---
Cost of Risk / Loans (%) | 1.25 | 1.06 | 0.93 | 0.89
Loan-to-Deposit Ratio (%) | 89 | 89 | 90 | 90
"""

# Define the golden reference output

In [12]:
from textwrap import dedent

reference_output = dedent("""
Executive Summary
Over the 2021–2024 period, BIAT demonstrates a consistent and well-balanced financial performance, combining solid revenue growth, improving efficiency, and controlled risk metrics. The bank gradually transitions from a high-growth phase toward a more mature and optimized operating model, while maintaining strong capital and liquidity buffers.

Overall, BIAT stands out as a high-quality banking franchise within the Tunisian market, with resilient fundamentals and disciplined execution.

Profitability and Operational Efficiency
Profitability has strengthened steadily across the period. Net income increased from TND 280 million in 2021 to TND 420 million in 2024, translating into a return on equity rising from 15.6% to approximately 17.5%.

This improvement is primarily driven by two structural factors. First, revenue growth has remained robust, supported by expanding lending activity and stable funding conditions. Second, the bank has achieved progressive efficiency gains, with the cost-to-income ratio declining from 49.5% to 45.5%.

This combination of top-line growth and cost discipline indicates that BIAT is benefiting from operating leverage, likely supported by digitalization initiatives and scale effects. At current levels, profitability is approaching best-in-class standards for comparable emerging market banks.

Revenue Dynamics
Net Banking Income (NBI) grew consistently over the period, increasing from TND 1,050 million in 2021 to TND 1,450 million in 2024. While growth remains strong, there is a visible moderation in momentum, with annual growth rates declining from above 12% to below 10%.

Loan growth, averaging around 9–10% annually, remains the primary driver, complemented by a stable and expanding deposit base. This suggests that BIAT continues to benefit from core commercial banking activity, with a balanced contribution between volume expansion and margin stability.

The gradual slowdown in growth indicates a transition toward a more mature phase, where efficiency and optimization become more important than pure expansion.

Asset Quality and Risk Profile
The bank’s risk profile shows overall improvement, with early signs of stabilization. The non-performing loan (NPL) ratio declined from 8.5% in 2021 to 7.6% in 2023, before slightly increasing to 7.8% in 2024.

At the same time, provisioning discipline strengthened, with the coverage ratio reaching around 70%. The cost of risk decreased steadily as a percentage of loans, reflecting improved credit quality and effective risk management practices.

The slight uptick in NPLs in 2024 may signal emerging macroeconomic pressures, but at this stage it remains contained and does not materially alter the overall risk trajectory.

Balance Sheet Structure and Liquidity
BIAT maintains a sound and conservative balance sheet structure. Loan growth is consistently funded by deposits, with the loan-to-deposit ratio remaining stable at around 90% throughout the period.

Liquidity metrics are strong, with the Liquidity Coverage Ratio (LCR) consistently above 120%, indicating a comfortable buffer against short-term stress scenarios.

This profile reflects a traditional and resilient funding model, limiting reliance on volatile wholesale funding and reinforcing the bank’s stability.

Capital Adequacy
Capitalization has improved gradually, with the CET1 ratio increasing from 11.5% in 2021 to 12.5% in 2024. This evolution is mainly driven by organic capital generation through retained earnings.

At these levels, BIAT maintains a solid buffer above regulatory requirements, providing flexibility to support future growth, absorb potential shocks, or sustain dividend distributions.

Key Risks and Watch Points
Despite the overall strong performance, several risk factors should be monitored closely:

- Exposure to the Tunisian macroeconomic environment and sovereign risk
- Potential credit concentration in certain corporate sectors
- Risk of margin compression in a changing interest rate environment
- The slight deterioration in asset quality observed in 2024, which may signal early stress

Conclusion
BIAT’s financial profile over 2021–2024 reflects a well-managed and fundamentally strong institution. The bank combines:

- Sustained profitability
- Continuous efficiency improvements
- Controlled credit risk
- Strong capital and liquidity positions

As growth naturally moderates, the key differentiator going forward will be BIAT’s ability to maintain efficiency gains and manage emerging risks in a more constrained macroeconomic environment.

Overall, the bank remains well-positioned, resilient, and structurally sound, with a moderate but sustainable growth outlook.
""").strip()

## REFERENCE sections CHECK

In [13]:
import re
from textwrap import dedent

EXPECTED_SECTIONS = [
    "Executive Summary",
    "Profitability and Operational Efficiency",
    "Revenue Dynamics",
    "Asset Quality and Risk Profile",
    "Balance Sheet Structure and Liquidity",
    "Capital Adequacy",
    "Key Risks and Watch Points",
    "Conclusion",
]

def normalize_report_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.strip()
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[–—−]", "-", text)
    # Remove bold markdown formatting (**text** -> text)
    text = re.sub(r"\*\*(.+?)\*\*", r"\1", text)
    return text

def extract_sections_rule_based(text: str) -> dict[str, str]:
    text = normalize_report_text(text)
    results = {name: "" for name in EXPECTED_SECTIONS}
    matches = []

    for section in EXPECTED_SECTIONS:
        pattern = rf"(?im)^[ \t]*(?:#+[ \t]*)?{re.escape(section)}[ \t]*:?[ \t]*$"
        for m in re.finditer(pattern, text):
            matches.append((m.start(), m.end(), section))

    matches.sort(key=lambda x: x[0])

    seen = set()
    ordered = []
    for item in matches:
        key = (item[0], item[2])
        if key not in seen:
            ordered.append(item)
            seen.add(key)

    for i, (_, end_pos, section) in enumerate(ordered):
        next_start = ordered[i + 1][0] if i + 1 < len(ordered) else len(text)
        results[section] = text[end_pos:next_start].strip()

    return results

ref_sections = extract_sections_rule_based(reference_output)

print("REFERENCE CHECK")
for k, v in ref_sections.items():
    print(k, "->", "OK" if v.strip() else "EMPTY")

assert all(v.strip() for v in ref_sections.values()), "Reference output is missing one or more sections."

REFERENCE CHECK
Executive Summary -> OK
Profitability and Operational Efficiency -> OK
Revenue Dynamics -> OK
Asset Quality and Risk Profile -> OK
Balance Sheet Structure and Liquidity -> OK
Capital Adequacy -> OK
Key Risks and Watch Points -> OK
Conclusion -> OK


# Create  the MLflow evaluation dataset

In [14]:
from mlflow.genai.datasets import create_dataset

DATASET_NAME = "bank_financial_analysis_golden_v4"
dataset = create_dataset(
    name=DATASET_NAME,
    experiment_id=[experiment.experiment_id],
    tags={
        "project": "slm-evals",
        "task": "bank_financial_analysis",
        "status": "active",
        "version": "1.0",
    },
)

print("Dataset ID:", dataset.dataset_id)
print("Dataset name:", dataset.name)

Dataset ID: d-8437d4ddf8124fc2a3c9e6d0f5c96fbd
Dataset name: bank_financial_analysis_golden_v4


# Add the first golden record

In [15]:
golden_records = [
    {
        "inputs": {
            "case_id": "test_case_004",
            "financial_data": financial_input_text,
        },
        "expectations": {
            "expected_response": reference_output,
        },
        "tags": {
            "split": "golden"
        }
    }
]

dataset = dataset.merge_records(golden_records)

dataset_df = dataset.to_df()
print("Dataset record count:", len(dataset_df))
display(dataset_df)

Dataset record count: 1


,inputs,outputs,expectations,tags,source_type,source_id,source,created_time,dataset_record_id
0,"{'case_id': 'test_case_004', 'financial_data':...",{},{'expected_response': 'Executive Summary Over ...,"{'split': 'golden', 'mlflow.user': 'BRHN'}",HUMAN,None,DatasetRecordSource(source_type=<DatasetRecord...,1777153863901,dr-64a56f73cb7a4f39b786dde53e9ac736


In [16]:
dataset_df.columns.tolist()
display(dataset_df[["inputs", "expectations", "tags"]])

,inputs,expectations,tags
0,"{'case_id': 'test_case_004', 'financial_data':...",{'expected_response': 'Executive Summary Over ...,"{'split': 'golden', 'mlflow.user': 'BRHN'}"


# Log notebook artifacts and metadata into the bootstrap run

In [17]:
dataset_df = dataset.to_df()
golden_record_count = len(dataset_df)

with mlflow.start_run(run_id=BOOTSTRAP_RUN_ID):
    mlflow.log_params({
        "prompt_name": PROMPT_NAME,
        "prompt_version": prompt_version.version,
        "prompt_uri_baseline": PROMPT_URI_BASELINE,
        "prompt_uri_versioned": PROMPT_URI_VERSIONED,
        "dataset_name": DATASET_NAME,
        "dataset_id": dataset.dataset_id,
        "golden_record_count":  golden_record_count,
    })

    mlflow.log_text(json.dumps(PROMPT_TEMPLATE, indent=2), "prompt/prompt_template.json")
mlflow.log_text(reference_output, "golden/reference_output.txt")
mlflow.log_text(financial_input_text, "golden/financial_input.txt")

🏃 View run nb2_bootstrap_setup at: http://127.0.0.1:5000/#/experiments/4/runs/e7eb046d032a409fa8f3404047bddea4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [18]:
dataset_df = dataset.to_df()
golden_record_count = len(dataset_df)

summary = {
    "tracking_uri": mlflow.get_tracking_uri(),
    "experiment_name": experiment.name,
    "experiment_id": experiment.experiment_id,
    "bootstrap_run_id": BOOTSTRAP_RUN_ID,
    "prompt_name": PROMPT_NAME,
    "prompt_version": prompt_version.version,
    "prompt_uri_baseline": PROMPT_URI_BASELINE,
    "prompt_uri_versioned": PROMPT_URI_VERSIONED,
    "dataset_name": DATASET_NAME,
    "dataset_id": dataset.dataset_id,
    "dataset_records":  golden_record_count,
    "notebook_2_ready": True,
}

print(json.dumps(summary, indent=2))

{
  "tracking_uri": "http://127.0.0.1:5000",
  "experiment_name": "phi4_vs_Llama3-1_baseline",
  "experiment_id": "4",
  "bootstrap_run_id": "e7eb046d032a409fa8f3404047bddea4",
  "prompt_name": "financial_analysis_baseline",
  "prompt_version": 10,
  "prompt_uri_baseline": "prompts:/financial_analysis_baseline@baseline",
  "prompt_uri_versioned": "prompts:/financial_analysis_baseline/10",
  "dataset_name": "bank_financial_analysis_golden_v4",
  "dataset_id": "d-8437d4ddf8124fc2a3c9e6d0f5c96fbd",
  "dataset_records": 1,
  "notebook_2_ready": true
}


In [19]:
assert experiment is not None
assert prompt_version is not None
assert dataset is not None
assert len(dataset_df) > 0
assert "expected_response" in str(dataset_df["expectations"].iloc[0])

print("Notebook 1 validation passed.")

Notebook 1 validation passed.
